# Fase 2-Colab — Multi-seed: 150 Epoch Langsung (Auto-resume)

**Proyek:** DeMamba-YOLOWD
**Prasyarat:** `fase-setup-colab-mamba.ipynb` sudah divalidasi penuh.
**Menggantikan:** `fase-2a-colab...` + `fase-2b-colab...` (2 notebook terpisah, skema 100+50)
untuk seed 1-4 — **karena seed baru tidak punya checkpoint 100-epoch yang perlu disambung**,
beda dengan seed 0 yang memang sudah kadung dilatih 100 epoch duluan.

## Kenapa 150 epoch sekali target (bukan 100+50) lebih baik untuk seed baru

Skema 100+50 di seed 0 (dan semua notebook Kaggle 2A/2B) **BUKAN pilihan ideal** — itu
keharusan, karena B2/P1 seed-0 sudah terlanjur dilatih 100 epoch sebelum kita sadar butuh 150
untuk perbandingan adil. Konsekuensinya: LR scheduler **restart** di titik sambung, bukan
jadwal kontinu tunggal (dicatat sbg keterbatasan berulang di semua notebook 1D/1E/2A/2B).

Untuk seed 1-4, tidak ada keharusan itu — bisa langsung target **150 epoch, LR schedule
tunggal dari awal sampai akhir**, metodologis lebih bersih.

## ⚠️ Konsekuensi yang perlu masuk metodologi paper

Ini menciptakan **perbedaan metodologi antar-seed**: seed 0 = 100+50 (LR-restart), seed 1-4 =
150 kontinu. Bukan cacat fatal (B2 vs P1 tetap dapat perlakuan **identik** di dalam seed yang
sama, itu yang menjaga keadilan perbandingan inti), tapi WAJIB ditulis eksplisit di metodologi
paper. Pertimbangkan juga apakah seed 0 layak diulang bersih (150 kontinu) demi konsistensi
penuh — dicatat sebagai keputusan terbuka, bukan diputuskan di sini.

## Desain: auto-resume kalau checkpoint belum lengkap

Notebook ini **aman dijalankan ulang berkali-kali** untuk kombinasi yang sama (mis. kalau sesi
Colab terputus di tengah jalan, entah dapat GPU cepat atau lambat):
- **Belum ada checkpoint** → mulai training baru, target 150 epoch
- **Ada checkpoint, TAPI belum selesai** (`optimizer` masih ada, `epoch` tersimpan >= 0 dan
  < 149) → **resume=True** otomatis, lanjut dari titik terakhir — genuine resume, LR schedule
  kontinu, TIDAK ada restart
- **Ada checkpoint, SUDAH selesai** (`optimizer=None`, sudah lewat `final_eval`) → tidak
  training ulang, langsung laporkan hasil

Jadi: GPU kencang → selesai satu sesi. GPU lambat/sesi kepotong → tinggal jalankan ulang
notebook yang sama, otomatis lanjut dari titik terakhir, tanpa perlu edit apa pun.

## Cara pakai

1. Edit **Sel 4**: `MODEL`, `SEED`
2. **Runtime → Run all**
3. Kalau sesi terputus sebelum selesai: buka lagi, **Run all** lagi — otomatis lanjut
4. Setelah benar-benar selesai (epoch 150), catat `SAVE_DIR` untuk arsip

## Progres

**5 seed RESMI = seed1-5** (seed0 dikecualikan dari uji statistik, diarsip terpisah — lihat tracker).

| # | Model | Seed | Status |
|---|---|---|---|
| 1 | B2 | 1 | ✅ |
| 2 | B2 | 2 | ✅ |
| 3 | B2 | 3 | ✅ |
| 4 | B2 | 4 | ⬜ *(sedang jalan)* |
| 5 | B2 | 5 | ⬜ |
| 6 | P1 | 1 | ✅ |
| 7 | P1 | 2 | ✅ |
| 8 | P1 | 3 | ✅ |
| 9 | P1 | 4 | ✅ |
| 10 | P1 | 5 | ⬜ |


## 1. Mount Drive + Kredensial Kaggle (Cache)

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os, stat, shutil

DRIVE_ROOT = "/content/drive/MyDrive/demamba-yolowd"
os.makedirs(DRIVE_ROOT, exist_ok=True)
KAGGLE_JSON_CACHE = f"{DRIVE_ROOT}/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)
assert os.path.exists(KAGGLE_JSON_CACHE), (
    f"kaggle.json tidak ada di {KAGGLE_JSON_CACHE} -- jalankan fase-setup-colab-mamba.ipynb dulu."
)
shutil.copy(KAGGLE_JSON_CACHE, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", stat.S_IRUSR | stat.S_IWUSR)
print("kaggle.json dimuat dari cache Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
kaggle.json dimuat dari cache Drive.


## 2. Tarik Dataset dari Kaggle + Deteksi GPU

In [11]:
import subprocess, sys, glob, torch

# Kontrol numerik lintas-GPU: cudnn.allow_tf32 default TRUE di torch, tapi TF32 cuma ada
# di hardware Ampere+ (A100, L4) -- TIDAK ADA di Turing (T4). Artinya B2/P1 seed0 (Kaggle T4)
# dan seed1+ (Colab, GPU bisa A100/L4) TIDAK menjalankan presisi numerik yang identik kalau
# dibiarkan default -- variabel yang belum terkontrol. Dimatikan di sini demi konsistensi
# lintas-platform, ditemukan setelah B2 seed1 (A100) turun di 7/8 kelas vs seed0 (T4).
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print(f"TF32 dimatikan (matmul={torch.backends.cuda.matmul.allow_tf32}, cudnn={torch.backends.cudnn.allow_tf32})")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/deform_src", exist_ok=True)

subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/dsf-yolo-steel-tube-dataset",
                 "-p", "/content/data", "--unzip"], check=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/deform-ss2d-py",
                 "-p", "/content/deform_src", "--unzip"], check=True)

DEFORM_SRC = glob.glob("/content/deform_src/**/deform_ss2d.py", recursive=True)[0]
DSF_BASE = "/content/data/steel-tube-dataset-all/yolo"
assert os.path.exists(f"{DSF_BASE}/images/train2021")
print(f"DEFORM_SRC = {DEFORM_SRC}\nDSF_BASE   = {DSF_BASE}")

print(f"\nGPU: {torch.cuda.get_device_name(0)}  (compute cap {torch.cuda.get_device_capability(0)})")
print("(GPU cepat -> mungkin selesai 1 sesi. GPU lambat/T4 -> auto-resume kalau kepotong, lihat Sel 7)")

TF32 dimatikan (matmul=False, cudnn=False)
DEFORM_SRC = /content/deform_src/deform_ss2d.py
DSF_BASE   = /content/data/steel-tube-dataset-all/yolo

GPU: NVIDIA A100-SXM4-40GB  (compute cap (8, 0))
(GPU cepat -> mungkin selesai 1 sesi. GPU lambat/T4 -> auto-resume kalau kepotong, lihat Sel 7)


## 3. 🔧 KONFIGURASI — Edit Ini Setiap Kali Ganti Kombinasi

In [12]:
# ============================================================
#  UBAH BARIS INI SEBELUM "Run all"
# ============================================================
MODEL = "B2"   # "B2" atau "P1"
SEED  = 5      # 1, 2, 3, 4, atau 5. Set 5 seed RESMI utk uji statistik = seed1-5 (BUKAN 0-4) --
               # seed0 (data lama, resep beda/100+50) sengaja dikecualikan, diarsip sbg konteks
               # historis saja, TIDAK diulang & TIDAK ikut uji signifikansi final.
LR0 = 0.002   # WAJIB diturunkan dari default 0.01 -- lr0=0.01 terbukti tidak stabil utk P1+STE.
              # Bukti: linear rusak @epoch82 (LR saat itu ~0.0046), kosinus rusak LEBIH CEPAT
              # @ epoch7 (LR saat itu ~0.00995, msh dekat lr0 penuh -- kosinus menahan LR tinggi
              # lebih lama di awal, bukan menurunkannya lebih cepat spt diduga sebelumnya).
              # Ambang bahaya empiris ~0.0046-0.0054. lr0=0.002 dimulai DI BAWAH ambang itu
              # sejak epoch 1, bukan menurun ke arahnya -- margin aman, bukan mepet.
USE_COS_LR = False   # KEMBALI ke linear -- kosinus TERBUKTI LEBIH BURUK (rusak epoch 7 vs 70+),
                     # krn kurvanya datar di awal (menahan LR dekat lr0 lebih lama), padahal yang
                     # dibutuhkan adalah LR rendah SEJAK AWAL, bukan cuma di akhir. Linear + LR0
                     # rendah menyerang akar masalah (nilai LR), bukan cuma bentuk kurvanya.
# ============================================================

TARGET_EPOCHS = 150

assert MODEL in ("B2", "P1"), f"MODEL harus 'B2' atau 'P1', dapat: {MODEL}"
assert SEED in (1, 2, 3, 4, 5), f"SEED harus 1-5, dapat: {SEED}"

RUN_NAME = f"{MODEL.lower()}_seed{SEED}_150ep"
if USE_COS_LR:
    RUN_NAME += "_coslr"
if LR0 != 0.01:
    RUN_NAME += f"_lr{LR0}"
# nama run beda kalau konfigurasi beda -- supaya TIDAK ke-resume checkpoint dgn train_args lama
# yang tidak sesuai (mis. lr0/cos_lr beda), harus mulai training dari nol tiap konfigurasi baru
CONFIG_YAML = "Mamba-YOLO-T.yaml" if MODEL == "B2" else "DeMamba-YOLO-T.yaml"
EXPECTED_PARAMS = 5.987 if MODEL == "B2" else 5.931
EXPECTED_NDEF = 0 if MODEL == "B2" else 10
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f"MODEL={MODEL}  SEED={SEED}  RUN_NAME={RUN_NAME}  target={TARGET_EPOCHS} epoch  lr0={LR0}  cos_lr={USE_COS_LR}")

MODEL=B2  SEED=5  RUN_NAME=b2_seed5_150ep_lr0.002  target=150 epoch  lr0=0.002  cos_lr=False


## 4. Clone Mamba-YOLO + Semua Patch + Build Kernel

Identik Fase 2A-Colab/2B-Colab -- 8 patch registrasi + 2 patch `weights_only` (`tasks.py`) +
1 patch `weights_only` (`torch_utils.py`), kernel vendor-only.

In [13]:
import re, py_compile

REPO = "/content/Mamba-YOLO"
# WAJIB sebelum rmtree: pastikan cwd proses TIDAK sedang di dalam REPO (bisa terjadi kalau
# notebook di-run ulang di kernel yang sama setelah sel pre-flight sempat os.chdir(REPO) --
# menghapus folder yang sedang "ditempati" proses bikin SEMUA subprocess berikutnya gagal
# dgn "fatal: Unable to read current working directory").
os.chdir("/content")
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HZAI-ZJNU/Mamba-YOLO.git", REPO], check=True)

DEFORM = f"{REPO}/ultralytics/nn/modules/deform_ss2d.py"
shutil.copy(DEFORM_SRC, DEFORM)

src = open(DEFORM).read()
checks = {
    "STE (gradien Delta-p)": "x_soft" in src,
    "ssm_ratio=1.4 (capacity match P1)": "ssm_ratio=1.4" in src,
    "subclass block terdaftar": "class DeformVSSBlock" in src and "class DeformXSSBlock" in src,
}
if not all(checks.values()):
    raise RuntimeError("deform_ss2d.py BUKAN versi final -- cek dataset 'deform-ss2d-py' di Kaggle.")
print("Tripwire deform_ss2d.py OK")

def patch(path, subs):
    p = f"{REPO}/{path}"
    src = open(p).read()
    for old, new, tag in subs:
        if new in src:
            print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new)
        print(f"  OK {tag}")
    open(p, "w").write(src)

print("\n### Patch")
patch("ultralytics/nn/modules/mamba_yolo.py", [
    ("class XSSBlock(nn.Module):\n    def __init__(",
     "class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.1"),
    ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,",
     "nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,", "1.2"),
    ("class VSSBlock(nn.Module):\n    def __init__(",
     "class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.3"),
    ("            self.op = SS2D(",
     "            self.op = self.SSM_CLASS(", "1.4"),
])
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("from .mamba_yolo import SS2D",
     "from .mamba_yolo import SS2D, VSSBlock, XSSBlock", "2.1"),
])
patch("ultralytics/nn/tasks.py", [
    ("    XSSBlock\n)\n",
     "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import DeformVSSBlock, DeformXSSBlock\n", "3.1"),
    ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
     "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock\n        }:", "3.2"),
    ("RepC3, XSSBlock}:",
     "RepC3, XSSBlock, DeformXSSBlock}:", "3.3"),
    ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
     '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:', "4.1"),
    ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
     '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)', "4.2"),
])
patch("ultralytics/utils/torch_utils.py", [
    ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
     '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:', "5.1"),
])

y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
y = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", y)
y = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", y)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml", "w").write(y)

for f in ["ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py",
          "ultralytics/nn/tasks.py", "ultralytics/utils/torch_utils.py"]:
    py_compile.compile(f"{REPO}/{f}", doraise=True)
print("  OK sintaks semua file valid")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"], check=True)
nproc = int(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip() or "2")
os.environ["MAX_JOBS"] = str(max(1, nproc))

# Cache wheel selective_scan ke Drive, ditandai arsitektur GPU + versi torch/python --
# ini satu-satunya langkah yang genuinely lambat (kompilasi CUDA). Clone+patch di atas
# sudah cepat & SELALU dijalankan fresh (supaya deform_ss2d.py terbaru & patch selalu
# terverifikasi) -- yang di-cache cuma hasil compile-nya, bukan langkah patch-nya.
cap = torch.cuda.get_device_capability(0)
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
SS_WHEEL_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"
os.makedirs(SS_WHEEL_DIR, exist_ok=True)

cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
if cached_ss:
    print(f"Wheel selective_scan cache ditemukan ({SS_TAG}) -- install langsung, skip compile")
    subprocess.run([sys.executable, "-m", "pip", "install"] + cached_ss, check=True)
else:
    print(f"Belum ada cache utk {SS_TAG} -- compile sekali ini (~beberapa menit), lalu disimpan ke Drive")
    subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-v", "-w", SS_WHEEL_DIR],
                   cwd=f"{REPO}/selective_scan", check=True)
    new_ss_wheels = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    subprocess.run([sys.executable, "-m", "pip", "install"] + new_ss_wheels, check=True)
    print(f"Tersimpan ke {SS_WHEEL_DIR} -- sesi berikutnya dgn GPU {SS_TAG.split('_')[0]} langsung cepat")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)

for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]:
    del sys.modules[k]
import selective_scan_cuda_core
print("selective_scan_cuda_core berhasil di-import.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn", "thop", "timm", "einops"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "wandb", "ray", "tensorboard", "tensorboardX", "tensorboard-data-server",
                "clearml", "comet_ml", "dvclive", "mlflow", "neptune"])

yaml_path = f"{REPO}/ultralytics/cfg/datasets/dsf-yolo.yaml"
yaml_content = f"""path: {DSF_BASE}
train: images/train2021
val: images/val2021

nc: 8
names: [\'air-hole\', \'bite-edge\', \'broken-arc\', \'crack\', \'dense-gas-hole\', \'non-fusion\', \'slag-inclusion\', \'unfused-hole\']
"""
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("\nSEMUA PATCH + BUILD OK.")

Tripwire deform_ss2d.py OK

### Patch
  OK 1.1
  OK 1.2
  OK 1.3
  OK 1.4
  ~ 2.1: skip
  OK 3.1
  OK 3.2
  OK 3.3
  OK 4.1
  OK 4.2
  OK 5.1
  OK sintaks semua file valid
Wheel selective_scan cache ditemukan (sm80_torch2.11.0_py312) -- install langsung, skip compile
selective_scan_cuda_core berhasil di-import.

SEMUA PATCH + BUILD OK.


## 5. Pre-flight — Verifikasi Arsitektur

In [14]:
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from ultralytics import settings
settings.update({"tensorboard": False})

from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules.deform_ss2d import DeformSS2D
from ultralytics.nn.modules.mamba_yolo import SS2D

cfg_path = f"ultralytics/cfg/models/mamba-yolo/{CONFIG_YAML}"
m = DetectionModel(cfg_path, ch=3, nc=8, verbose=False)
n_par = sum(x.numel() for x in m.parameters())
n_def = sum(1 for x in m.modules() if isinstance(x, DeformSS2D))
n_plain = sum(1 for x in m.modules() if isinstance(x, SS2D) and not isinstance(x, DeformSS2D))

print(f"MODEL={MODEL}  SEED={SEED}  config={CONFIG_YAML}")
print(f"Params: {n_par/1e6:.3f}M (target {EXPECTED_PARAMS}M)  DeformSS2D: {n_def} (target {EXPECTED_NDEF})")
assert abs(n_par/1e6 - EXPECTED_PARAMS) < 0.01 and n_def == EXPECTED_NDEF, "Arsitektur tidak sesuai. STOP."

del m
torch.cuda.empty_cache()
print("Pre-flight LOLOS.")

Overriding model.yaml nc=80 with nc=8
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
WARNING ⚠️ no model scale passed. Assuming scale='T'.
MODEL=B2  SEED=5  config=Mamba-YOLO-T.yaml
Params: 5.987M (target 5.987M)  DeformSS2D: 0 (target 0)
Pre-flight LOLOS.


## 6. Cek Status Checkpoint — Baru / Belum Selesai / Sudah Selesai

Sel ini yang membuat notebook aman dijalankan ulang. Tidak ada training di sini, cuma
pengecekan status.

In [15]:
from pathlib import Path

last_ckpt = Path(DRIVE_OUTPUT) / RUN_NAME / "weights" / "last.pt"

if not last_ckpt.exists():
    STATUS = "BARU"
    print(f"STATUS: {STATUS} -- belum ada checkpoint, training akan mulai dari nol.")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep = raw.get("epoch", -1)
    has_opt = raw.get("optimizer") is not None
    if has_opt and ep >= 0 and ep < TARGET_EPOCHS - 1:
        STATUS = "BELUM_SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint ada, epoch tersimpan={ep}, optimizer ada.")
        print(f"  -> Akan RESUME (genuine, LR schedule kontinu) dari epoch {ep+1}.")
    else:
        STATUS = "SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint sudah final_eval (optimizer={has_opt}, epoch={ep}).")
        print("  -> Training TIDAK diulang. Lanjut ke Sel 8 untuk baca hasil.")

print(f"\nlast.pt: {last_ckpt}  (ada: {last_ckpt.exists()})")

STATUS: BARU -- belum ada checkpoint, training akan mulai dari nol.

last.pt: /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/last.pt  (ada: False)


## 7. Training — Mulai Baru atau Resume, Sesuai Status Sel 6

**`patience=0` WAJIB** — mematikan EarlyStopping. Proyek ini butuh jumlah epoch yang
**terkontrol ketat** (150, sama persis tiap run) untuk perbandingan B2 vs P1 yang adil.
Default Ultralytics `patience=100` membuat run berhenti otomatis begitu 100 epoch berlalu
tanpa perbaikan `fitness` — kalau dibiarkan, run yang berbeda bisa berhenti di jumlah epoch
berbeda-beda (satu model plateau lebih cepat dari yang lain), merusak prinsip
compute-matched/epoch-matched yang jadi dasar seluruh perbandingan proyek ini.

In [ ]:
if STATUS == "BARU":
    model = YOLO(cfg_path)
    model.train(
        data="ultralytics/cfg/datasets/dsf-yolo.yaml",
        epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0,   # WAJIB: matikan EarlyStopping, perbandingan butuh epoch terkontrol ketat
        cache=True,   # percepat: dataset (~2700 gambar) muat di RAM, hindari re-decode JPEG tiap
                      # epoch. TIDAK mengubah data/hasil -- murni percepatan, aman utk perbandingan.
        optimizer="SGD",  # WAJIB: resep asli seed0 (mbyolo_train.py) pakai SGD, defaultnya.
                          # Tanpa ini, Ultralytics diam-diam pilih optimizer=auto (AdamW+LR
                          # auto) -- ditemukan sbg penyebab UTAMA divergensi P1 (bukan platform).
        cos_lr=USE_COS_LR,  # False (default eksperimen ini) -- kosinus TERBUKTI LEBIH BURUK,
                            # lihat catatan lengkap di Sel 3.
        lr0=LR0,  # WAJIB diturunkan dari default 0.01 -- lihat catatan lengkap di Sel 3.
                  # Ini yang menyerang akar masalah (nilai LR), bukan cuma bentuk kurvanya.
        # JANGAN PERNAH set time= -- itu override epochs= secara diam-diam (sama bahayanya dgn patience)
        project=DRIVE_OUTPUT, name=RUN_NAME,
    )
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt))
    # TANPA epochs=/patience=/optimizer= -- resume mengambil train_args tersimpan di checkpoint
    # (150, patience=0, optimizer=SGD -- asalkan run aslinya sudah pakai settingan ini di atas).
    model.train(resume=True)
else:
    model = YOLO(str(last_ckpt))
    print("Training sudah selesai sebelumnya -- sel ini di-skip, tidak training ulang.")

SAVE_DIR = model.trainer.save_dir if hasattr(model, "trainer") and model.trainer else Path(DRIVE_OUTPUT) / RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")

# Verifikasi status akhir -- kalau sesi ini SENDIRI terputus, baris ini tidak akan tercetak,
# dan menjalankan ulang notebook (dari Sel 1) akan mendeteksi STATUS=BELUM_SELESAI otomatis.
final_check = torch.load(str(Path(SAVE_DIR) / "weights" / "last.pt"), map_location="cpu", weights_only=False)
if final_check.get("optimizer") is None:
    print(f"Training {RUN_NAME} SELESAI PENUH ({TARGET_EPOCHS} epoch).")
else:
    print(f"Training belum selesai (epoch {final_check.get('epoch')}) -- jalankan ulang notebook ini untuk lanjut.")

WARNING ⚠️ no model scale passed. Assuming scale='T'.
WARNING ⚠️ no model scale passed. Assuming scale='T'.
New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Ultralytics YOLOv8.2.29 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml, data=ultralytics/cfg/datasets/dsf-yolo.yaml, epochs=150, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/content/drive/MyDrive/demamba-yolowd/output, name=b2_seed5_150ep_lr0.002, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=5, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mos

train: Scanning /content/data/steel-tube-dataset-all/yolo/labels/train2021.cache... 2726 images, 6 backgrounds, 0 corrupt: 100%|██████████| 2726/2726 [00:00<?, ?it/s]
train: Caching images (2.2GB RAM): 100%|██████████| 2726/2726 [00:02<00:00, 1026.27it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



/content/Mamba-YOLO/ultralytics/data/augment.py:935: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /content/data/steel-tube-dataset-all/yolo/labels/val2021.cache... 682 images, 2 backgrounds, 0 corrupt: 100%|██████████| 682/682 [00:00<?, ?it/s]
val: Caching images (0.5GB RAM): 100%|██████████| 682/682 [00:00<00:00, 938.06it/s]


Plotting labels to /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/labels.jpg... 
Plotting labels to /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/labels.jpg... 
optimizer: SGD(lr=0.002, momentum=0.937) with parameter groups 71 weight(decay=0.0), 168 weight(decay=0.0005), 166 bias(decay=0.0)
optimizer: SGD(lr=0.002, momentum=0.937) with parameter groups 71 weight(decay=0.0), 168 weight(decay=0.0005), 166 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002
Starting training for 150 epochs...
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        682       1381      0.854      0.851      0.903      0.543
                   all        682       1381      0.854      0.851      0.903      0.543

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    150/150      5.71G      1.537      1.053      1.214          6        640: 100%|██████████| 341/341 [00:57<00:00,  5.96it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:04<00:00,  9.04it/s]


                   all        682       1381      0.862      0.855      0.903      0.539
                   all        682       1381      0.862      0.855      0.903      0.539

150 epochs completed in 2.616 hours.

150 epochs completed in 2.616 hours.
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/last.pt, 12.3MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/last.pt, 12.3MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/best.pt, 12.3MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/best.pt, 12.3MB

Validating /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/best.pt...

Validating /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/best.pt...
Ultralytics YOLOv8.2.29 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:05<00:00,  7.90it/s]


                   all        682       1381      0.855      0.851      0.904      0.543
                   all        682       1381      0.855      0.851      0.904      0.543
              air-hole        406       1070       0.91      0.586      0.846      0.353
              air-hole        406       1070       0.91      0.586      0.846      0.353
             bite-edge          8          8      0.923          1      0.995      0.604
             bite-edge          8          8      0.923          1      0.995      0.604
            broken-arc         74         74      0.702      0.795       0.79      0.582
            broken-arc         74         74      0.702      0.795       0.79      0.582
                 crack         23         23      0.597      0.652      0.699      0.391
                 crack         23         23      0.597      0.652      0.699      0.391
        dense-gas-hole         43         43      0.937      0.884      0.975      0.582
        dense-gas-hol

## 8. Baca Hasil (Hanya Kalau Sudah Selesai Penuh)

In [ ]:
import pandas as pd

df = pd.read_csv(Path(SAVE_DIR) / "results.csv")
df.columns = df.columns.str.strip()
last = df.iloc[-1]

print(f"=== {RUN_NAME} — {len(df)} baris tercatat (epoch terakhir: {int(df['epoch'].max())}) ===")
print(f"  mAP50    = {last['metrics/mAP50(B)']:.4f}")
print(f"  mAP50:95 = {last['metrics/mAP50-95(B)']:.4f}")
print(f"  P={last['metrics/precision(B)']:.4f}  R={last['metrics/recall(B)']:.4f}")

# Cek epoch TERAKHIR yang tercapai, BUKAN jumlah baris -- results.csv kadang bolong di
# tengah (mis. Drive I/O sempat gagal nulis 1-2 baris) walau training-nya sendiri utuh
# tanpa gangguan. len(df) < TARGET_EPOCHS bisa salah nunjuk "belum selesai" padahal
# training sudah genap sampai epoch target -- ditemukan dari kasus B2 seed4 (146 baris
# tapi epoch max=150, gap di epoch 26-27 & 87-88, loss mulus tanpa lompatan di sekitarnya).
epoch_tercapai = int(df['epoch'].max())
n_gap = TARGET_EPOCHS - len(df)
if epoch_tercapai < TARGET_EPOCHS:
    print(f"\n⚠️  Epoch terakhir cuma {epoch_tercapai}/{TARGET_EPOCHS} -- training BELUM selesai.")
    print("   Jalankan ulang notebook ini (Runtime > Run all) untuk lanjut.")
else:
    print(f"\nSELESAI PENUH -- epoch terakhir tercapai {epoch_tercapai}/{TARGET_EPOCHS}.")
    if n_gap > 0:
        missing = sorted(set(range(1, TARGET_EPOCHS+1)) - set(df['epoch'].astype(int)))
        print(f"  Catatan: {n_gap} baris log bolong di tengah (epoch {missing}) -- kemungkinan")
        print(f"  Drive I/O sempat gagal nulis, BUKAN training terputus. Cek loss di sekitar")
        print(f"  epoch itu kalau ragu (harus mulus, tanpa lompatan/reset).")
    print(f"Checkpoint: {Path(SAVE_DIR) / 'weights' / 'best.pt'}")
    print("Catat path ini untuk arsip & analisis agregat multi-seed nanti.")


=== b2_seed5_150ep_lr0.002 — 150 baris tercatat (epoch terakhir: 150) ===
  mAP50    = 0.9027
  mAP50:95 = 0.5391
  P=0.8620  R=0.8547

SELESAI PENUH -- epoch terakhir tercapai 150/150.
Checkpoint: /content/drive/MyDrive/demamba-yolowd/output/b2_seed5_150ep_lr0.002/weights/best.pt
Catat path ini untuk arsip & analisis agregat multi-seed nanti.


## 9. Breakdown Per-Kelas

Angka agregat (Sel 8) tidak cukup untuk Ablasi F (apakah keuntungan P1 terkonsentrasi di kelas
elongated/diagonal seperti crack, walau kalah di rata-rata karena air-hole yang dominan jumlahnya).
Sel ini WAJIB dijalankan tiap run selesai -- jangan cuma andalkan angka agregat Sel 8.

In [ ]:
import pandas as pd

model_final = YOLO(str(Path(SAVE_DIR) / "weights" / "best.pt"))
mf = model_final.val(data="ultralytics/cfg/datasets/dsf-yolo.yaml", split="val", verbose=False)

print(f"=== {RUN_NAME} -- per-kelas mAP50:95 ===")
print(f"Agregat: mAP50={mf.box.map50:.4f}  mAP50:95={mf.box.map:.4f}\n")
print(f"{'Kelas':16s} {'mAP50:95':>9s}")
for ci, cn in mf.names.items():
    print(f"{cn:16s} {mf.box.maps[ci]:>9.4f}")

# Simpan ke CSV (append, bukan timpa) -- MODEL & SEED diambil OTOMATIS dari variabel
# Sel 3, bukan diketik ulang manual (itu sumber human-error sebelumnya: lupa update
# nomor seed saat pakai ulang snippet utk kombinasi lain).
OUT_CSV = f"{DRIVE_ROOT}/analysis/per_class_results.csv"
Path(f"{DRIVE_ROOT}/analysis").mkdir(parents=True, exist_ok=True)

rows = [{
    "model": MODEL, "seed": SEED, "class": "ALL",
    "mAP50_95": mf.box.map, "mAP50": mf.box.map50,
    "precision": mf.box.mp, "recall": mf.box.mr,
    "ckpt_path": str(Path(SAVE_DIR) / "weights" / "best.pt"),
}]
for ci, cn in mf.names.items():
    rows.append({
        "model": MODEL, "seed": SEED, "class": cn,
        "mAP50_95": mf.box.maps[ci], "mAP50": None,
        "precision": None, "recall": None,
        "ckpt_path": str(Path(SAVE_DIR) / "weights" / "best.pt"),
    })
df_new = pd.DataFrame(rows)

if Path(OUT_CSV).exists():
    df_old = pd.read_csv(OUT_CSV)
    df_out = pd.concat([df_old, df_new], ignore_index=True)
    df_out = df_out.drop_duplicates(subset=["model", "seed", "class"], keep="last")
else:
    df_out = df_new
df_out.to_csv(OUT_CSV, index=False)
print(f"\nTersimpan ke: {OUT_CSV}  ({len(df_out)} baris total, {MODEL} seed{SEED})")


Ultralytics YOLOv8.2.29 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Ultralytics YOLOv8.2.29 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Mamba-YOLO-T summary: 388 layers, 5985568 parameters, 0 gradients, 13.6 GFLOPs
Mamba-YOLO-T summary: 388 layers, 5985568 parameters, 0 gradients, 13.6 GFLOPs


val: Scanning /content/data/steel-tube-dataset-all/yolo/labels/val2021.cache... 682 images, 2 backgrounds, 0 corrupt: 100%|██████████| 682/682 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:06<00:00,  6.36it/s]


                   all        682       1381      0.855      0.852      0.905      0.543
                   all        682       1381      0.855      0.852      0.905      0.543
Speed: 0.1ms preprocess, 5.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Speed: 0.1ms preprocess, 5.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/Mamba-YOLO/runs/detect/val
Results saved to /content/Mamba-YOLO/runs/detect/val
=== b2_seed5_150ep_lr0.002 -- per-kelas mAP50:95 ===
Agregat: mAP50=0.9049  mAP50:95=0.5428

Kelas             mAP50:95
air-hole            0.3529
bite-edge           0.6005
broken-arc          0.5852
crack               0.3939
dense-gas-hole      0.5818
non-fusion          0.6314
slag-inclusion      0.5258
unfused-hole        0.6708

Tersimpan ke: /content/drive/MyDrive/demamba-yolowd/analysis/per_class_results.csv  (90 baris total, B2 seed5)


In [ ]:
# Matikan runtime Colab otomatis setelah semua sel di atas selesai --
# supaya TIDAK terus memakan kuota Colab Pro kalau kamu tidak sempat
# matikan manual. Ini API resmi Google, benar-benar melepas VM-nya.
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)  # jeda singkat, jaga-jaga ada file yang masih proses ditulis ke Drive

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")

## Checklist

- [ ] Sel 1-2: Drive mount, dataset Kaggle, GPU terdeteksi
- [ ] Sel 3: konfigurasi MODEL/SEED benar
- [ ] Sel 4-5: patch + pre-flight LOLOS
- [ ] Sel 6: status checkpoint terbaca (BARU / BELUM_SELESAI / SELESAI)
- [ ] Sel 7: training jalan sesuai status (baru/resume/skip)
- [ ] Sel 8: **150/150 epoch** — kalau belum, ulangi Sel 1-8 (Run all) sampai selesai
- [ ] Sel 9: breakdown per-kelas tercatat (WAJIB, jangan skip -- dibutuhkan utk Ablasi F nanti)
- [ ] Centang baris di tabel progres (Sel 0)

**Catatan metodologi untuk paper:** seed 1-4 (notebook ini) = 150 epoch kontinu, LR schedule
tunggal. Seed 0 (Kaggle, sudah ada) = 100+50 dengan LR-restart. Perbedaan ini WAJIB
disebutkan eksplisit di bagian metodologi — pertimbangkan apakah seed 0 perlu diulang bersih
untuk konsistensi penuh sebelum menulis draft final.